In [ ]:
# this notebook will prepare data to visualize on chart
# create markers
import json
import pandas as pd
import gc
from pathlib import Path
import numpy as np

DATA_PATH = Path.cwd().parents[1] / "data" / "bigData" / "BTCUSD-15m.json"

with open(DATA_PATH) as f:
    file = json.load(f)

ohlcv = file["ohlcv"]  # list of [timestamp, open, high, low, close, volume]
del file  # free the rest of the JSON (symbol, onchart, etc.), , preserve RAM

df = pd.DataFrame(ohlcv, columns=["timestamp", "open", "high", "low", "close", "vol"])
del ohlcv  # free the Python list-of-lists, preserve RAM
gc.collect()

# Halve memory: float64 -> float32 for price/volume columns
df[["open", "high", "low", "close", "vol"]] = df[["open", "high", "low", "close", "vol"]].astype("float32")

# print(df.head())
# beware  was truncated by the NotebookEdit tool on the previous write
print(f"Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum()/ 1e6:.2f} MB")

Shape: (70930, 6) | Memory: 1.99 MB


In [3]:
df.head()
# save as JSONL
# OUT_PATH = Path.cwd().parents[1] / "data" / "bigData" / "BTCUSD-15m.jsonl"
# df.to_json(OUT_PATH, orient="records", lines=True)
# print(f"Saved {len(df)} rows to {OUT_PATH}")

,timestamp,open,high,low,close,vol
0,1708659900000,51099.929688,51120.000000,51052.199219,51100.000000,209.838379
1,1708660800000,51100.000000,51106.871094,51003.000000,51091.859375,230.937881
2,1708661700000,51091.871094,51236.640625,51031.101562,51236.640625,333.208618
3,1708662600000,51236.628906,51321.921875,51183.210938,51224.031250,284.106201
4,1708663500000,51224.039062,51254.199219,51195.839844,51219.589844,169.352463


# Common Methods for Normalizing OHLC Data
- Min-Max Scaling (Rolling Window)
- Log Returns / Percentage Change
- Standardization (Z-score)
- Base-Price Normalization : divide all by base

In [5]:
def _fix_break_of_structure(isXXX, isSource, high, low):
    """
    Ensure isXXX alternates strictly between 1 and -1.
    On a break (1,1 or -1,-1), insert the best counter swing between the two offending positions,
    but ONLY if it is structurally valid:
      - Inserting a  1 (high): candidate high must be > max(low[p1],  low[p2])
      - Inserting a -1 (low) : candidate low  must be < min(high[p1], high[p2])
    If no valid candidate exists, remove the less extreme of the two duplicates instead.
    """
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i - 1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 != v2:
                continue

            counter    = np.int8(-v1)
            candidates = np.arange(p1 + 1, p2)

            inserted = False
            if len(candidates) > 0:
                source_cands = candidates[isSource[candidates] == counter]
                pool = source_cands if len(source_cands) > 0 else candidates

                if counter == -1:
                    # Inserting a low: must dip below both surrounding highs
                    threshold  = min(high[p1], high[p2])
                    valid_pool = pool[low[pool] < threshold]
                else:
                    # Inserting a high: must rise above both surrounding lows
                    threshold  = max(low[p1], low[p2])
                    valid_pool = pool[high[pool] > threshold]

                if len(valid_pool) > 0:
                    best = (valid_pool[np.argmin(low[valid_pool])]
                            if counter == -1
                            else valid_pool[np.argmax(high[valid_pool])])
                    isXXX[best] = counter
                    inserted = True

            if not inserted:
                # No structurally valid candidate — drop the less extreme duplicate
                if v1 == 1:   # two highs: keep the higher one
                    isXXX[p1 if high[p1] < high[p2] else p2] = 0
                else:          # two lows: keep the lower one
                    isXXX[p1 if low[p1]  > low[p2] else p2] = 0

            changed = True
            break  # restart after each change

    return isXXX


In [6]:
def _fix_price_order(isXXX, high, low):
    """
    Fix price ordering violations by inserting 2 bridging marks instead of removing.
    For ITR/LTR: inserted marks need not be original swing points.
    Falls back to removal only if no valid bridge candidates exist.
    """
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i-1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 == 1 and v2 == -1 and low[p2] >= high[p1]:
                # Need a real low then a real high between p1 and p2
                cands = np.arange(p1 + 1, p2)
                valid_lows = cands[low[cands] < high[p1]]
                if len(valid_lows) > 0:
                    best_low = valid_lows[np.argmin(low[valid_lows])]
                    after = np.arange(best_low + 1, p2)
                    valid_highs = after[high[after] > low[best_low]]
                    if len(valid_highs) > 0:
                        best_high = valid_highs[np.argmax(high[valid_highs])]
                        isXXX[best_low]  = -1
                        isXXX[best_high] = 1
                        changed = True
                        break
                # fallback: remove the bad low
                isXXX[p2] = 0
                changed = True
                break

            elif v1 == -1 and v2 == 1 and high[p2] <= low[p1]:
                # Need a real high then a real low between p1 and p2
                cands = np.arange(p1 + 1, p2)
                valid_highs = cands[high[cands] > low[p1]]
                if len(valid_highs) > 0:
                    best_high = valid_highs[np.argmax(high[valid_highs])]
                    after = np.arange(best_high + 1, p2)
                    valid_lows = after[low[after] < high[best_high]]
                    if len(valid_lows) > 0:
                        best_low = valid_lows[np.argmin(low[valid_lows])]
                        isXXX[best_high] = 1
                        isXXX[best_low]  = -1
                        changed = True
                        break
                # fallback: remove the bad high
                isXXX[p2] = 0
                changed = True
                break

    return isXXX


In [ ]:
# varsion 1
def strHunt(df, window=4):
    n = len(df)
    df = df.copy()

    high = df['high'].values
    low  = df['low'].values

    isSTR = np.zeros(n, dtype=np.int8)

    # --- First window: can't look backward, so determine direction from the span ---
    # if n > window + 1:
    #     diff = high[window + 1] - low[0]
    #     if diff > 0:
    #         isSTR[0] = -1   # start of bullish move → mark as low
    #     elif diff < 0:
    #         isSTR[0] = 1    # start of bearish move → mark as high

    # --- First window: method 2 just mark 0

    # --- Main loop: candles with a full symmetric window on both sides ---
    # Stop at n-window so the last `window` candles remain 0
    for i in range(window, n - window):
        win_high = high[i - window : i + window + 1]
        win_low  = low [i - window : i + window + 1]

        is_swing_high = high[i] >= win_high.max()   # local maximum
        is_swing_low  = low[i]  <= win_low.min()    # local minimum

        if is_swing_high and not is_swing_low:
            isSTR[i] = 1    # confirmed swing high
        elif is_swing_low and not is_swing_high:
            isSTR[i] = -1   # confirmed swing low
        # neither → stays 0
        # both, liquidity swept on both high and low → stays 0
        

    isSTR = _fix_break_of_structure(isSTR, isSTR, high, low)
    isSTR = _fix_price_order(isSTR, high, low)

    df['isSTR'] = isSTR
    return df

# execute -- 
df = strHunt(df)

In [7]:
def itrHunt(df, group=3):
    # there must be isSTR column presented first
    # search for intermediate term range
    # group params input must be odd number (has middle point)
    # to qualify, the middle swing high must be the highest among `group` consecutive swing HIGHS
    # to qualify, the middle swing low must be the lowest among `group` consecutive swing LOWS
    # highs are compared against highs only, lows against lows only
    # (requires at least 1,1,1 or -1,-1,-1 in isSTR for a valid group comparison)

    df = df.copy()
    n = len(df)

    high  = df['high'].values
    low   = df['low'].values
    isSTR = df['isSTR'].values
    isITR = np.zeros(n, dtype=np.int8)

    # Separate by type — compare like with like
    high_pos = np.where(isSTR == 1)[0]   # all swing high positions
    low_pos  = np.where(isSTR == -1)[0]  # all swing low positions

    half = group // 2  # e.g. group=3 → half=1

    # ITR swing highs: middle must be highest among `group` consecutive swing highs
    for j in range(half, len(high_pos) - half):
        group_pos = high_pos[j - half : j + half + 1]
        mid_pos   = high_pos[j]
        if high[mid_pos] >= high[group_pos].max():
            isITR[mid_pos] = 1

    # ITR swing lows: middle must be lowest among `group` consecutive swing lows
    for j in range(half, len(low_pos) - half):
        group_pos = low_pos[j - half : j + half + 1]
        mid_pos   = low_pos[j]
        if low[mid_pos] <= low[group_pos].min():
            isITR[mid_pos] = -1

    # handle break of structure case
    isITR = _fix_break_of_structure(isITR, isSTR, high, low)
    # handle exception case : price ordering
    isITR = _fix_price_order(isITR, high, low)


    df['isITR'] = isITR
    return df


# execute --
df = itrHunt(df)

In [8]:
def ltrHunt(df, group=3):
    # there must be isITR column presented first
    # search for long term range
    # highs are compared against highs only, lows against lows only
    # (requires at least 1,1,1 or -1,-1,-1 in isITR for a valid group comparison)

    df = df.copy()
    n = len(df)

    high  = df['high'].values
    low   = df['low'].values
    isITR = df['isITR'].values
    isLTR = np.zeros(n, dtype=np.int8)

    # Separate by type — compare like with like
    high_pos = np.where(isITR == 1)[0]
    low_pos  = np.where(isITR == -1)[0]

    half = group // 2

    for j in range(half, len(high_pos) - half):
        group_pos = high_pos[j - half : j + half + 1]
        mid_pos   = high_pos[j]
        if high[mid_pos] >= high[group_pos].max():
            isLTR[mid_pos] = 1

    for j in range(half, len(low_pos) - half):
        group_pos = low_pos[j - half : j + half + 1]
        mid_pos   = low_pos[j]
        if low[mid_pos] <= low[group_pos].min():
            isLTR[mid_pos] = -1

    # handle break of structure case
    isLTR = _fix_break_of_structure(isLTR, isITR, high, low)
    # handle exception case : price ordering
    isLTR = _fix_price_order(isLTR, high, low)


    df['isLTR'] = isLTR
    return df


# execute --
df = ltrHunt(df)

In [9]:
# check point - preview data
df.tail()

,timestamp,open,high,low,close,vol,isSTR,isITR,isLTR
70925,1772492400000,69440.476562,69479.578125,69215.398438,69223.242188,121.096069,0,0,0
70926,1772493300000,69223.250000,69261.687500,68990.976562,69002.929688,223.747665,0,0,0
70927,1772494200000,69002.921875,69014.257812,68758.398438,68796.859375,195.868637,0,0,0
70928,1772495100000,68796.859375,68965.671875,68740.000000,68830.062500,97.017410,0,0,0
70929,1772496000000,68830.062500,69004.398438,68750.679688,68809.992188,168.970734,0,0,0


In [ ]:
def validate_markers(df, col):
    """
    Validate a marker column (e.g. 'isITR', 'isLTR') across the entire DataFrame.

    Checks two invariants:
      1. BOS (Break-of-Structure): consecutive non-zero entries must alternate sign (no 1,1 or -1,-1).
      2. Price order: after a high (1), the next low (-1) must be strictly below it,
         and after a low (-1), the next high (1) must be strictly above it.

    Returns
    -------
    dict with keys:
        'bos_violations'   : DataFrame of rows where sign repeats
        'order_violations' : DataFrame of rows where price order is broken
        'is_valid'         : True only if both checks pass
    """
    rows = df[df[col] != 0][['timestamp', 'high', 'low', col]].copy()
    rows['price_at_mark'] = rows.apply(
        lambda r: r['high'] if r[col] == 1 else r['low'], axis=1
    )

    vals = rows[col].values

    # --- BOS check: consecutive same-sign entries ---
    bos_mask = [False] + [vals[i] == vals[i - 1] for i in range(1, len(vals))]
    bos_violations = rows[bos_mask]

    # --- Price order check ---
    order_fail_idx = []
    for i in range(1, len(rows)):
        prev = rows.iloc[i - 1]
        curr = rows.iloc[i]
        if curr[col] == 1:          # current is a high → must be above prev low
            if curr['high'] <= prev['low']:
                order_fail_idx.append(rows.index[i])
        else:                        # current is a low → must be below prev high
            if curr['low'] >= prev['high']:
                order_fail_idx.append(rows.index[i])
    order_violations = rows.loc[order_fail_idx]

    # --- Print summary ---
    label = col
    total = len(rows)
    print(f"=== {label} validation ({total} marks across full dataset) ===")

    if len(bos_violations):
        print(f"  BOS violations : {len(bos_violations)}")
        print(bos_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  BOS violations : 0  — alternation OK")

    if len(order_violations):
        print(f"  Order violations: {len(order_violations)}")
        print(order_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  Order violations: 0  — price ordering OK")

    is_valid = len(bos_violations) == 0 and len(order_violations) == 0
    print(f"  Valid: {is_valid}\n")

    return {
        'bos_violations':   bos_violations,
        'order_violations': order_violations,
        'is_valid':         is_valid,
    }


=== isSTR validation (10886 marks across full dataset) ===
  BOS violations : 2048
           timestamp           high            low  isSTR  price_at_mark
38     1708694100000   51042.429688   50925.019531     -1   50925.019531
50     1708704900000   50877.839844   50629.328125     -1   50629.328125
78     1708730100000   50805.578125   50521.000000     -1   50521.000000
94     1708744500000   50782.710938   50682.000000     -1   50682.000000
131    1708777800000   51065.140625   51032.339844     -1   51032.339844
145    1708790400000   51166.000000   51094.988281     -1   51094.988281
170    1708812900000   51666.000000   51605.558594      1   51666.000000
182    1708823700000   51513.539062   51438.878906     -1   51438.878906
195    1708835400000   51561.101562   51509.000000     -1   51509.000000
210    1708848900000   51777.640625   51729.988281      1   51777.640625
221    1708858800000   51627.179688   51553.289062     -1   51553.289062
229    1708866000000   51693.589844   515

In [11]:
# --- run validation on STR columns ---
str_report = validate_markers(df, 'isSTR')


=== isSTR validation (10886 marks across full dataset) ===
  BOS violations : 2048
           timestamp           high            low  isSTR  price_at_mark
38     1708694100000   51042.429688   50925.019531     -1   50925.019531
50     1708704900000   50877.839844   50629.328125     -1   50629.328125
78     1708730100000   50805.578125   50521.000000     -1   50521.000000
94     1708744500000   50782.710938   50682.000000     -1   50682.000000
131    1708777800000   51065.140625   51032.339844     -1   51032.339844
145    1708790400000   51166.000000   51094.988281     -1   51094.988281
170    1708812900000   51666.000000   51605.558594      1   51666.000000
182    1708823700000   51513.539062   51438.878906     -1   51438.878906
195    1708835400000   51561.101562   51509.000000     -1   51509.000000
210    1708848900000   51777.640625   51729.988281      1   51777.640625
221    1708858800000   51627.179688   51553.289062     -1   51553.289062
229    1708866000000   51693.589844   515

In [12]:
# --- run validation on ITR columns ---
itr_report = validate_markers(df, 'isITR')

=== isITR validation (3644 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True



In [13]:
# --- run validation on LTR columns ---
ltr_report = validate_markers(df, 'isLTR')

=== isLTR validation (1169 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True



In [ ]:
# Export marker to json visulize the data on Chart

# onchart data
```json
onchart : [{
    name: "Marker",
    type: "Marker",
    data: marking,
    settings: {
        "z-index": 5,
        "legend": false
    }
}]
```

where marking is

```json
[
     1771612200000,  // timestamp
     'buy',          // marker type: 'buy' | 'sell' | 'str' | 'itr' | 'ltr'
     67527.01,       // value at mark
     "lo"            // legend (optional)
], ...
```



In [ ]:
def exportJsonWithMarker(df, out_path):
    itr_df = df[df['isITR'] != 0][['timestamp', 'isITR', 'high', 'low']]

    data = [
        [
            int(row['timestamp']),
            'itr',
            float(row['high']) if row['isITR'] == 1 else float(row['low']),
            ''
        ]
        for _, row in itr_df.iterrows()
    ]

    payload = {
        "onchart": [{
            "name": "Marker",
            "type": "Marker",
            "data": data,
            "settings": {
                "z-index": 5,
                "legend": False
            }
        }]
    }

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w') as f:
        json.dump(payload, f)

    print(f"Exported {len(data)} markers to {out_path}")


# execute
OUT_PATH = Path.cwd().parents[1] / "data" / "bigData" / "15m-marker.json"
exportJsonWithMarker(df, OUT_PATH)
